# 01 — EDA: cross-sector retention panel

**Project:** Dynamic Retention Benchmarking (H03) · **Author:** Asad Kamran · MADS, University of Michigan · Dubai HR

An organisation cannot tell whether 84% retention is good or bad without knowing the *sector-conditional* benchmark and the action that would lift it. This notebook profiles the synthetic 8-sector panel from `retention_bench.data.generate()` so the per-sector Ridge regression and the contextual bandit downstream both rest on a checked distribution.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from retention_bench.data import generate, write_processed, PROCESSED, SECTORS, ACTIONS
from retention_bench.features import add_engineered

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

In [ ]:
PARQUET = PROCESSED / 'retention_panel.parquet'
if PARQUET.exists():
    df = pd.read_parquet(PARQUET)
else:
    df = generate()
    write_processed(df)
df = add_engineered(df)
df.shape, df.columns.tolist()

## 1. Sector-level retention distribution

The headline output is a per-sector retention distribution. Public-sector organisations sit at the top, hospitality and retail at the bottom — that is the structure built into the synthetic generator (`SECTOR_BASE`).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
order = df.groupby('sector')['retention_rate'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='sector', y='retention_rate', order=order, ax=ax)
ax.axhline(df['retention_rate'].median(), color='red', ls='--', alpha=0.7, label='global median')
ax.set_title('Retention rate distribution by sector')
ax.legend()
plt.tight_layout(); plt.show()

## 2. Predictor distributions

Compensation percentile, training hours, manager quality. The Ridge models in `03_model.ipynb` use these directly; we audit their distributions for skew and outliers.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, col in zip(axes, ['comp_percentile', 'training_hours', 'manager_quality']):
    sns.histplot(df, x=col, bins=30, ax=ax, color='#3a7ca5')
    ax.set_title(col)
plt.tight_layout(); plt.show()

## 3. Headcount distribution — log-scale inspection

Headcount is log-normal by construction; we want to confirm it before feeding raw `headcount` to the Ridge.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
sns.histplot(df, x='log_headcount', bins=30, ax=ax, color='#4a7c59')
ax.set_xlabel('log(1+headcount)')
ax.set_title('Headcount on the log scale')
plt.tight_layout(); plt.show()

## 4. Retention vs each predictor — within-sector trends

Ridge fits per sector mean we expect non-trivial slope variation across sectors. A scatter conditioned on `sector` makes that variation visible.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.scatterplot(data=df, x='manager_quality', y='retention_rate', hue='sector', alpha=0.45, ax=ax)
ax.set_title('Retention vs manager_quality, conditioned on sector')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout(); plt.show()

## 5. Action-mix per sector

The bandit's job is to recommend *which action* to take. The dataset's empirical action mix is roughly uniform with a `no_action` plurality — that is what the bandit pulls against.

In [ ]:
action_mix = df.groupby(['sector', 'action_taken']).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(10, 4))
action_mix.plot(kind='bar', stacked=True, ax=ax, colormap='tab10')
ax.set_title('Action mix per sector (synthetic)')
ax.set_ylabel('# org-year rows')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout(); plt.show()

## 6. Reward signal by action × sector

Per-sector mean reward by action — the table the bandit's `rank()` will recover at convergence.

In [ ]:
reward = df.groupby(['sector', 'action_taken'])['reward'].mean().unstack(fill_value=np.nan).round(4)
fig, ax = plt.subplots(figsize=(8.5, 4))
sns.heatmap(reward, annot=True, fmt='.3f', cmap='RdYlGn', center=0, ax=ax, cbar_kws={'label': 'mean reward'})
ax.set_title('Mean reward by sector × action')
plt.tight_layout(); plt.show()

## 7. Hypotheses for `03_model.ipynb`

1. **Per-sector Ridge will out-perform a single global Ridge** — sector-specific intercepts and slopes are large enough that pooling shrinks signal.
2. **`manager_quality` is the strongest universal predictor** of retention rate; comp percentile matters more in private-sector slices.
3. **The bandit's exploit-only ranking will agree with the empirical reward heatmap above** for each sector — that agreement is the operational sanity check.